# Unix-terminal. Настройка окружения и установка пакетов


## Мотивация

Python-проект — это не только код, но и версия интерпретатора, точные версии пакетов и их зависимостей. Пакет с тем же именем после обновления может вести себя иначе: например, код вокруг `timm` ожидает четыре промежуточные карты признаков модели, а получает три и ломается уже во время обучения. Отдельное окружение и зафиксированные зависимости позволяют воспроизвести рабочий запуск на другой машине и обновлять проект осознанно.


## 1. Переменные оболочки, `export` и `source`

`NAME=value` создаёт переменную в текущей оболочке. `export NAME=value` добавляет её в окружение: значение получают программы и дочерние оболочки, запущенные после `export`. Запись `NAME=value command` передаёт значение только процессу `command`.

`source FILE` выполняет команды из файла в текущей оболочке, поэтому смена каталога и значения переменных сохраняются. `bash FILE` запускает отдельную дочернюю оболочку: её переменные исчезнут после завершения, а родительская оболочка не изменится.

Ни обычная переменная, ни `export` не сохраняются после закрытия оболочки: новый терминал запускает новый процесс shell. Постоянные настройки записывают в стартовый файл Bash. Интерактивный Bash, запущенный не как login shell, читает `~/.bashrc`; login shell — первый доступный файл из `~/.bash_profile`, `~/.bash_login`, `~/.profile`. Какой вариант используется, зависит от способа запуска оболочки, поэтому незнакомый стартовый файл сначала читают.

`env` и `printenv` показывают окружение, которое получат дочерние процессы. Переменные без `export` в этот список не входят.


In [ ]:
%%bash
rm -rf ~/seminar-05/env       # чистый старт: следы прошлого запуска не мешают
mkdir -p ~/seminar-05/env

# в файле две переменные: одна обычная, вторая с export
cat > ~/seminar-05/env/course.env <<'EOF'
COURSE_NAME='course-app'
export COURSE_DATA="$HOME/seminar-05/env/data"
EOF

cat ~/seminar-05/env/course.env    # это просто текстовый файл с командами Bash


Файл создан, но пока ни на что не влияет: это текст. Теперь выполним его
в текущей оболочке через `source` и посмотрим, что достанется дочернему процессу.


In [ ]:
%%bash
source ~/seminar-05/env/course.env                 # source выполняет файл в текущей оболочке
echo "current: $COURSE_NAME"                       # обе переменные видны здесь
bash -c 'echo "child: ${COURSE_NAME:-missing}"'    # без export дочерний bash её не получит
bash -c 'echo "exported: $COURSE_DATA"'            # а экспортированную — получит
printenv COURSE_DATA                               # printenv показывает только окружение


#### ❓ **Вопрос**

В `course.env` переменная `COURSE_NAME` задана без `export`, а `COURSE_DATA` — с `export`. Что увидят текущая оболочка и новый `bash -c /usr/bin/env` после `source course.env`?

<details>
<summary>Ответ</summary>

Текущая оболочка увидит обе переменные, потому что `source` выполняет файл в ней. Дочерний `bash -c` получит только `COURSE_DATA`: без `export` значение `COURSE_NAME` не передаётся дочернему процессу.

</details>


## 2. `PATH` и `PYTHONPATH`

`PATH` — список каталогов с исполняемыми файлами, разделённых двоеточиями. Оболочка проверяет каталоги слева направо и запускает первый найденный файл. `which name` показывает выбранную команду.

Полный путь не требует поиска через `PATH`: `/usr/bin/ls` можно запустить даже при сломанном `PATH`. Это полезный временный обход, но постоянную причину всё равно исправляют.

`PYTHONPATH` добавляет каталоги в путь поиска Python-модулей. Для зависимостей проекта предпочтительнее `.venv`: глобальный `PYTHONPATH` легко подмешивает неожиданный код сразу в несколько проектов.

`env` выводит окружение процесса. Фильтр `env | grep '^COURSE_'` оставляет переменные, имена которых начинаются с `COURSE_`. Полный `env` нельзя публиковать: в нём могут находиться токены.


In [ ]:
%%bash
mkdir -p ~/seminar-05/bin
cat > ~/seminar-05/bin/course-info <<'EOF'
#!/usr/bin/env bash
echo "course=$COURSE_NAME"
EOF
chmod +x ~/seminar-05/bin/course-info    # без права x файл не запустить как команду

course-info || echo 'по имени команда не найдена: её каталога нет в PATH'


Файл исполняемый, но по имени не находится: оболочка ищет команды только
в каталогах из `PATH`. Добавим туда свой `bin`.


In [ ]:
%%bash
export PATH="$HOME/seminar-05/bin:$PATH"    # свой каталог в начало списка поиска
export COURSE_NAME='course-app'             # значение, которое прочитает сама команда

which course-info                 # теперь оболочка находит команду по имени
course-info                       # ...и запускает; COURSE_NAME пришло из окружения
env | grep '^COURSE_' || true     # что из COURSE_* реально попало в окружение


Добавили каталог — команда нашлась. А теперь противоположный случай:
`PATH` перезаписали целиком, потеряв системные каталоги.


In [ ]:
%%bash
PATH="$HOME/bin"    # так делать нельзя: старое значение PATH потеряно

# which — внешняя программа, её саму уже не найти; command -v встроен в оболочку
command -v ls || echo 'ls через PATH не находится'
/usr/bin/ls ~/seminar-05 >/dev/null && echo 'полный путь работает и без PATH'


#### ❓ **Вопрос**

После `PATH="$HOME/bin"` команда `ls` перестала находиться. Как вернуть поиск системных команд? Как однократно запустить `ls`, пока `PATH` ещё не исправлен?

<details>
<summary>Ответ</summary>

Нужно восстановить прежнее значение `PATH` или заново добавить системные каталоги. Когда к рабочему списку добавляют собственный каталог, используют `export PATH="$HOME/bin:$PATH"`, не отбрасывая старое значение. Если оно уже потеряно, `PATH` можно восстановить из стартового файла или новой оболочки. Для разового запуска используют полный путь, например `/usr/bin/ls`.

</details>


## 3. Интерпретатор, `-m`, `venv` и `virtualenv`


### Что делает `python -m`

Обычный запуск `python script.py` выполняет файл по указанному пути. Запуск `python -m module` просит **выбранный Python** найти модуль так же, как при импорте, и выполнить его как программу. Если указан обычный модуль, выполняется его `.py`-файл; если пакет — файл `__main__.py` внутри пакета. Отдельный модуль пакета запускают как `python -m package.module`.

Поэтому `python -m pip` запускает модуль `pip`, установленный именно у выбранного `python`, а не случайную одноимённую команду из другого места в `PATH`.


In [ ]:
%%bash
rm -rf ~/seminar-05/python-demo
mkdir -p ~/seminar-05/python-demo/course_app

# обычный модуль пакета — его запустит python -m course_app.say_hello_module
cat > ~/seminar-05/python-demo/course_app/say_hello_module.py <<'PY'
print("Hello world")
PY

# __main__.py — то, что выполняется при python -m course_app
cat > ~/seminar-05/python-demo/course_app/__main__.py <<'PY'
from . import say_hello_module
PY


Пакет `course_app` собран из двух файлов. Осталось показать его Python
через `PYTHONPATH` и запустить обоими способами.


In [ ]:
%%bash
# PYTHONPATH=… перед командой действует только на этот запуск
PYTHONPATH="$HOME/seminar-05/python-demo" python3 -m course_app                    # выполнится __main__.py пакета
PYTHONPATH="$HOME/seminar-05/python-demo" python3 -m course_app.say_hello_module   # выполнится отдельный модуль
# -m берёт модуль у того самого python, которым запускаем. В Ubuntu pip
# отдельный пакет (python3-pip) и в чистой системе его может не быть:
python3 -m pip --version || echo 'pip не установлен: sudo apt install python3-pip'


### Зачем несколько Python и несколько `.venv`

На одной машине могут одновременно требоваться разные версии Python: старый проект ещё работает с Python 3.10, новый использует возможности 3.12, а версия на рабочем сервере должна совпадать с проверенной. Виртуальное окружение создаётся **на основе конкретного интерпретатора** и не превращает Python 3.11 в Python 3.12.

Даже проекты на одной версии Python получают разные `.venv`: одному нужен пакет `library==1`, другому — несовместимый `library==2`. Глобальная или пользовательская установка смешивает зависимости проектов: обновление одного пакета может сломать соседний код, а рабочее окружение будет трудно воспроизвести на другой машине.

```text
машина
├── system Python 3.11          ← нужен ОС и системным утилитам
├── Python 3.12 под управлением uv
│   ├── project-a/.venv         ← requests 2.31
│   └── project-b/.venv         ← requests 2.32
└── Python 3.13 под управлением uv
    └── experiment/.venv        ← отдельный набор пакетов
```

`uv python find 3.12` показывает путь к подходящему интерпретатору. Активация `.venv` только добавляет её каталог `bin` в начало `PATH`. Окружение можно использовать и без активации — через `.venv/bin/python` или `uv run`.


In [ ]:
%%bash
rm -rf ~/seminar-05/demo-venv
uv venv --python 3.12 ~/seminar-05/demo-venv

~/seminar-05/demo-venv/bin/python -c \
  'import sys; print(sys.executable); print(sys.version)'

source ~/seminar-05/demo-venv/bin/activate
which python
python --version
deactivate


#### ❓ **Вопрос**

Проект A требует Python 3.11 и `library==1`, проект B — Python 3.12 и `library==2`. Достаточно ли двух `.venv`, созданных системным Python 3.11? Почему?

<details>
<summary>Ответ</summary>

Нет. Разные `.venv` изолируют версии `library`, но оба окружения останутся на Python 3.11. Для проекта B сначала нужен интерпретатор 3.12, а затем отдельная `.venv`, созданная на его основе.

</details>


## 4. Зависимости Python


### Прямые, транзитивные, ограничения и lock-файл

Допустим, код проекта импортирует `requests`:

- **прямая зависимость** — `requests`, потому что её выбрал сам проект и записал в `pyproject.toml`;
- **транзитивные зависимости** — например, `urllib3`, `certifi`, `idna`: они нужны `requests`, хотя проект не выбирал их напрямую;
- **ограничение версии** — `requests>=2.31,<3`: диапазон версий, которые разрешает проект, а не уже установленная версия;
- **зафиксированное решение** — точные версии всех выбранных пакетов и источники их загрузки, записанные в `uv.lock`.

```text
course-app
└── requests >=2.31,<3           ← прямая зависимость
    ├── urllib3                  ← транзитивная
    ├── certifi                  ← транзитивная
    └── idna                     ← транзитивная

pyproject.toml ── uv выбирает версии ──> uv.lock ── uv sync ──> .venv
  намерение                         точный план             файлы среды
```

Одинаковое имя пакета не гарантирует одинаковое поведение разных версий. Код может успешно импортировать обновлённый пакет, но получить другой результат или форму данных уже во время работы. Поэтому в Git сохраняют `pyproject.toml` и `uv.lock`, а `.venv` пересоздают.


### `uv` — основной инструмент курса

`uv` управляет Python, окружением и зависимостями проекта:

- `uv init` создаёт основу проекта;
- `uv python pin 3.12` записывает требуемый Python в `.python-version`;
- `uv add PACKAGE` добавляет прямую зависимость в `pyproject.toml`, устанавливает её и обновляет `uv.lock`;
- `uv remove PACKAGE` удаляет прямую зависимость из `pyproject.toml` и обновляет `uv.lock`;
- `uv tree` показывает дерево прямых и транзитивных зависимостей;
- `uv lock --check` проверяет, соответствует ли lock-файл описанию проекта;
- `uv sync` приводит `.venv` в состояние, записанное в `pyproject.toml` и `uv.lock`;
- `uv lock --upgrade` обновляет все пакеты до последних допустимых версий;
- `uv run COMMAND` синхронизирует окружение и запускает команду внутри него.

`uv.lock` не обновляется только потому, что в реестре появилась новая версия. Обновление выполняют явно.


In [ ]:
%%bash
rm -rf ~/seminar-05/project
mkdir -p ~/seminar-05/project
cd ~/seminar-05/project || exit 1

uv init                       # каркас проекта: pyproject.toml и служебные файлы
uv python pin 3.12            # требуемая версия интерпретатора — в .python-version
uv add 'requests>=2.31,<3'    # прямая зависимость: pyproject.toml + uv.lock + .venv


Проект создан. Сравним два файла: `pyproject.toml` хранит намерение
(диапазон версий), `uv.lock` — уже выбранное точное решение.


In [ ]:
%%bash
cd ~/seminar-05/project || exit 1

echo '--- pyproject.toml ---'
cat pyproject.toml    # намерение: диапазон версий, который разрешает проект

echo '--- fragment of uv.lock ---'
grep -n -A 12 'name = "requests"' uv.lock | head -n 20    # решение: точная версия и источник


Видно, что записано в файлах. Теперь посмотрим на дерево зависимостей
целиком и убедимся, что код действительно запускается в `.venv` проекта.


In [ ]:
%%bash
cd ~/seminar-05/project || exit 1

uv tree    # прямые зависимости и подтянутые за ними транзитивные
uv run python -c 'import sys, requests; print(sys.executable); print(requests.__version__)'


#### ❓ **Вопрос**

В `pyproject.toml` разрешено `requests>=2.31,<3`, а в `uv.lock` уже выбрана конкретная версия. В реестре появилась более новая совместимая версия. Поставит ли её обычный `uv sync` на другой машине?

<details>
<summary>Ответ</summary>

Нет. `uv sync` использует сохранённый lock-файл и воспроизводит зафиксированное решение. Чтобы перейти на новые совместимые версии, lock-файл обновляют через `uv lock --upgrade`.

</details>


## 5. Области установки и системные менеджеры

Выбор области зависит от того, кому нужна программа:

```text
Linux-система
├── system:  /usr/bin, /usr/lib       ← apt, dnf, pacman; для всех пользователей
├── user:    ~/.local/bin             ← личные CLI-инструменты, например uv tool
└── project: project/.venv            ← импортируемые зависимости конкретного проекта
```

- Библиотека, которую импортирует проект, добавляется в проект через `uv add`.
- Самостоятельную CLI-утилиту для одного пользователя можно установить через `uv tool install` или менеджер вроде Homebrew.
- Системная программа и её системные зависимости устанавливаются менеджером ОС: `apt`/`apt-get` в Debian и Ubuntu, `dnf` в Fedora/RHEL, `pacman` в Arch.

`apt update` обновляет локальный индекс доступных версий, а `apt upgrade` — установленные пакеты. `apt install` устанавливает пакет, `apt remove` удаляет его, `apt purge` дополнительно удаляет системные конфигурационные файлы пакета. `apt` удобен для интерактивной работы, а `apt-get` имеет более стабильный интерфейс для автоматизации.


In [ ]:
%%bash
mkdir -p ~/seminar-05/apt
apt list --installed \
  > ~/seminar-05/apt/installed-packages.txt \
  2> ~/seminar-05/apt/errors.txt

apt list --upgradable \
  > ~/seminar-05/apt/upgradable-packages.txt \
  2>> ~/seminar-05/apt/errors.txt

head -n 5 ~/seminar-05/apt/installed-packages.txt


#### ❓ **Вопрос**

Куда логичнее установить: `requests`, который импортирует один проект; `ruff`, используемый как личная CLI-утилита; системный `curl`? Когда `ruff` всё же стоит добавить в зависимости проекта?

<details>
<summary>Ответ</summary>

`requests` — в `.venv` проекта через `uv add`; личный `ruff` — как user tool; системный `curl` — менеджером ОС. Если команда и версия `ruff` должны быть одинаковыми у всей команды, его добавляют в группу зависимостей проекта.

</details>


## 6. Службы и журналы `systemd`

`systemd` запускает и контролирует фоновые службы системы. `systemctl` показывает их состояние и управляет запуском, а `journalctl` читает собранные журналы.

*`systemd-journald` — системная служба, которая собирает сообщения ядра, служб и других программ. Здесь она используется как пример, доступный почти в любой системе с systemd.*

Основные команды:

- `status NAME` — текущее состояние и несколько последних сообщений;
- `start`, `stop`, `restart` — управление сейчас;
- `enable NAME` — включить автоматический запуск при старте системы;
- `cat NAME` — показать найденный unit-файл и дополнения;
- `list-units --type=service` — загруженные службы;
- `list-unit-files --type=service` — все установленные unit-файлы служб;
- `journalctl -u NAME -n 50` — последние 50 сообщений службы;
- `--no-pager` — вывести результат прямо в терминал.

`start` запускает службу сейчас, но не включает автозапуск. `enable` включает автозапуск, но сам по себе не запускает службу. `enable --now` делает оба действия.


In [ ]:
%%bash
mkdir -p ~/seminar-05/systemd
service_name=systemd-journald

{
  systemctl status "$service_name" --no-pager
  systemctl cat "$service_name"
} > ~/seminar-05/systemd/service.txt 2> ~/seminar-05/systemd/errors.txt || true

journalctl -u "$service_name" -n 5 --no-pager \
  > ~/seminar-05/systemd/journal.txt 2>> ~/seminar-05/systemd/errors.txt || true


#### ❓ **Вопрос**

Что покажут `systemctl status systemd-journald`, `systemctl cat systemd-journald` и `journalctl -u systemd-journald`? Запустит ли `systemctl enable` остановленную службу немедленно?

<details>
<summary>Ответ</summary>

`status` показывает текущее состояние и последние сообщения, `cat` — найденный unit-файл и его дополнения, `journalctl -u` — журнал выбранной службы. `enable` только настраивает будущий автозапуск; немедленный запуск выполняют через `start` или `enable --now`.

</details>


## Дополнительно


### Старые и специализированные инструменты

`venv` входит в Python, `virtualenv` устанавливается отдельным пакетом. Оба создают окружение на основе выбранного интерпретатора; такие команды встречаются в старых проектах:

```bash
python3.12 -m venv .venv
python3 -m virtualenv --python=python3.12 .venv
python -m pip install -r requirements.txt
```

`conda` управляет окружениями, Python и пакетами из conda-каналов. Она полезна для проектов со сложными нативными зависимостями или уже существующим `environment.yml`:

```bash
conda create -n course python=3.12 requests
conda activate course
conda env export > environment.yml
```

Для обычных новых Python-проектов этого курса используем `uv`.


### Версии Python в `uv`

Команды решают разные задачи:

- `uv python install 3.12` — установить управляемый uv интерпретатор;
- `uv python pin 3.12` — записать требование проекта в `.python-version`;
- `uv python find 3.12` — показать выбранный путь;
- `uv python list` — показать найденные и доступные версии.

`pin` не создаёт `.venv` и не устанавливает зависимости.


### Обновление одной зависимости

`uv lock --upgrade-package requests` разрешает uv выбрать новую версию `requests` в пределах ограничений `pyproject.toml`. Остальные зафиксированные пакеты сохраняются, если обновление `requests` не требует их изменения. `uv sync` применяет новый lock-файл к `.venv`.

Перед обновлением полезно сохранить старый `uv.lock`, после — посмотреть `diff` и запустить проверки проекта.


### Подключение APT-репозитория

APT получает пакеты из настроенных источников. Стороннему источнику нужны адрес и ключ проверки подписи. На примере Docker для Ubuntu последовательность такая: установить `ca-certificates` и `curl`, сохранить ключ в `/etc/apt/keyrings`, создать `.sources`, обновить индекс и установить пакеты.

В `.sources`: `URIs` — адрес, `Suites` — версия Ubuntu, `Components` — ветка репозитория, `Architectures` — архитектура, `Signed-By` — ключ проверки подписи. Команды со стороннего сайта перед запуском сверяют с его актуальной официальной инструкцией.


> ⚠️ **Эти три блока — справочные, на занятии их не выполняют.** Они
> показаны как markdown, а не как ячейки, намеренно: `sudo` в ячейке ноутбука
> либо упадёт с «a terminal is required to read the password», либо, если
> пароль не спрашивают, действительно установит Docker на вашу машину. Читайте
> их как рецепт, а запускайте осознанно в обычном терминале.

```bash
sudo apt update                              # свежий индекс доступных версий
sudo apt install ca-certificates curl        # чем скачивать ключ и проверять TLS
sudo install -m 0755 -d /etc/apt/keyrings    # общий каталог для ключей репозиториев

sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg \
  -o /etc/apt/keyrings/docker.asc            # ключ, которым подписаны пакеты Docker
sudo chmod a+r /etc/apt/keyrings/docker.asc  # apt должен иметь право прочитать ключ
```

Ключ на месте. Теперь описываем сам источник: где брать пакеты, для
какой версии системы и каким ключом проверять подписи.


```bash
# $(...) подставляются локальной оболочкой: кодовое имя Ubuntu и архитектура машины
sudo tee /etc/apt/sources.list.d/docker.sources <<EOF
Types: deb
URIs: https://download.docker.com/linux/ubuntu
Suites: $(source /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}")
Components: stable
Architectures: $(dpkg --print-architecture)
Signed-By: /etc/apt/keyrings/docker.asc
EOF
```

Источник описан, но apt о нём ещё не знает: индекс собран до появления
файла. Обновляем индекс и ставим пакеты.


```bash
sudo apt update    # индекс теперь включает и пакеты нового источника

sudo apt install docker-ce docker-ce-cli containerd.io \
  docker-buildx-plugin docker-compose-plugin    # движок, CLI и плагины одной командой
```

### Что такое unit

Unit-файл — конфигурация systemd. Service-unit задаёт, какую команду запустить и как трактовать её выполнение:

```ini
[Unit]
Description=Show service user

[Service]
Type=oneshot
ExecStart=/usr/bin/id
```

`Type=oneshot` означает, что systemd запускает одно действие, ждёт его завершения и не ожидает постоянно работающий процесс. `ExecStart=/usr/bin/id` задаёт запускаемую команду абсолютным путём. Вывод команды попадает в журнал unit. `systemctl cat NAME.service` показывает основной unit-файл и его drop-in-дополнения — отдельные файлы, которые уточняют или переопределяют настройки. После изменения unit-файлов менеджер перечитывает их через `systemctl daemon-reload`.


### Системные и пользовательские службы

`systemctl --system` обращается к системному менеджеру systemd. Это режим по умолчанию: такие службы работают для всей машины, а изменение и управление ими обычно требуют прав администратора.

`systemctl --user` обращается к отдельному менеджеру текущего пользователя. Пользовательские unit-файлы обычно хранятся в `~/.config/systemd/user/`, не требуют `sudo` и управляют только процессами этого пользователя. Их журналы читают через `journalctl --user -u NAME`. Системная и пользовательская службы с одинаковым именем — разные unit, поэтому режим важно указывать последовательно.

После ручного запуска из терминала проект может не заработать как служба: systemd не запускает интерактивную оболочку, поэтому не читает пользовательский `~/.bashrc` и не выполняет `.venv/bin/activate`. Активация `.venv` лишь меняет `PATH` текущей оболочки и не влияет на отдельный процесс службы. Поэтому в unit-файле явно задают рабочий каталог, окружение и путь к интерпретатору:

```ini
[Service]
WorkingDirectory=/home/student/project
EnvironmentFile=/home/student/project/app.env
ExecStart=/home/student/project/.venv/bin/python /home/student/project/app.py
```

Для пользовательской службы, которая должна работать после выхода пользователя, может потребоваться lingering — разрешение оставлять пользовательский менеджер запущенным без активной сессии. Администратор включает его через `loginctl enable-linger USER`.
